# ENTSO-E Excel Price Data Processing

This notebook processes ENTSO-E price data from Excel and CSV files, cleans it, and saves to Parquet format.

**Input Files:**
- `GUI_ENERGY_PRICES_202412312300-202512312300 (1).xlsx` (2025 data)
- `Entsoe api\\GUI_ENERGY_PRICES_202601010000-202701010000.csv` (2026 data)

**Output:**
- `entsoe_prices_2025.parquet`
- `entsoe_prices_2026.parquet`
- Combined time series plot

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

def resolve_entsoe_path(path_str: str) -> Path:
    """Resolve notebook input paths from either repo root or the Entsoe api folder."""
    requested = Path(path_str)
    candidates = [requested, Path.cwd() / requested, Path.cwd() / requested.name]

    if requested.parts and requested.parts[0] == 'Entsoe api':
        candidates.append(Path(*requested.parts[1:]))

    candidates.extend([
        Path('Entsoe api') / requested.name,
        Path.cwd() / 'Entsoe api' / requested.name,
    ])

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve(strict=False)
        key = str(candidate).lower()
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find '{path_str}' from working directory {Path.cwd()}")

print("Libraries loaded")

Libraries loaded


## 1. Define Processing Function

Function to clean Excel data and extract timestamps and prices.

In [2]:
def process_entsoe_excel(excel_path: str, year: int) -> pd.DataFrame:
    """
    Process ENTSO-E Excel file and extract clean time series.
    
    Args:
        excel_path: Path to Excel file
        year: Year for the data (used for validation)
    
    Returns:
        DataFrame with columns: ts_utc (datetime), price (float in EUR/kWh)
    """
    excel_path = resolve_entsoe_path(excel_path)
    print(f"\nProcessing {excel_path}...")
    
    # Read Excel - skip header rows (metadata is in rows 1-7)
    df_raw = pd.read_excel(excel_path, skiprows=7)
    
    print(f"  Raw data shape: {df_raw.shape}")
    print(f"  Columns: {df_raw.columns.tolist()}")
    
    # Identify MTU column (time) and price column
    # MTU column should be first, price column should be second
    time_col = df_raw.columns[0]  # MTU
    price_col = df_raw.columns[1]  # BZN|NL Without Sequence Day-ahead (EUR/MWh)
    
    print(f"  Time column: '{time_col}'")
    print(f"  Price column: '{price_col}'")
    
    # Extract relevant columns
    df = df_raw[[time_col, price_col]].copy()
    df.columns = ['mtu', 'price']
    
    # Remove rows with missing values
    df = df.dropna()
    
    print(f"  After dropping NaN: {len(df)} rows")
    
    # Parse MTU column to extract start timestamp
    # Format: "01/01/2026 00:00:00 - 01/01/2026 00:15:00"
    def parse_mtu(mtu_str):
        try:
            # Extract start time (before the '-')
            start_str = mtu_str.split(' - ')[0].strip()
            # Parse as datetime
            return pd.to_datetime(start_str, format='%d/%m/%Y %H:%M:%S')
        except Exception as e:
            print(f"  Warning: Could not parse '{mtu_str}': {e}")
            return pd.NaT
    
    df['ts_utc'] = df['mtu'].apply(parse_mtu)
    
    # Remove rows where timestamp parsing failed
    df = df.dropna(subset=['ts_utc'])
    
    print(f"  After timestamp parsing: {len(df)} rows")
    
    # Convert to timezone-aware UTC
    # Note: ENTSO-E timestamps are typically in CET/CEST
    # For simplicity, we'll assume they're already in the correct timezone
    df['ts_utc'] = df['ts_utc'].dt.tz_localize('UTC', ambiguous='infer', nonexistent='shift_forward')
    
    # Convert price to numeric and handle any non-numeric values
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    
    # Remove rows with invalid prices
    df = df.dropna(subset=['price'])
    
    # Convert EUR/MWh to EUR/kWh
    df['price'] = df['price'] / 1000.0
    
    # Select final columns
    df_clean = df[['ts_utc', 'price']].copy()
    
    # Sort by timestamp
    df_clean = df_clean.sort_values('ts_utc').reset_index(drop=True)
    
    # Remove duplicates (keep last)
    df_clean = df_clean.drop_duplicates(subset=['ts_utc'], keep='last')
    
    print(f"  Final clean data: {len(df_clean)} rows")
    print(f"  Date range: {df_clean['ts_utc'].min()} to {df_clean['ts_utc'].max()}")
    print(f"  Price range: {df_clean['price'].min():.6f} - {df_clean['price'].max():.6f} EUR/kWh")
    
    # Validate year
    actual_years = df_clean['ts_utc'].dt.year.unique()
    print(f"  Years in data: {sorted(actual_years)}")
    
    return df_clean


def process_entsoe_csv(csv_path: str, year: int) -> pd.DataFrame:
    """
    Process ENTSO-E CSV file and extract clean time series.
    
    Args:
        csv_path: Path to CSV file
        year: Year for the data (used for validation)
    
    Returns:
        DataFrame with columns: ts_utc (datetime), price (float in EUR/kWh)
    """
    csv_path = resolve_entsoe_path(csv_path)
    print(f"\nProcessing {csv_path}...")
    
    # Read CSV
    df_raw = pd.read_csv(csv_path)
    
    print(f"  Raw data shape: {df_raw.shape}")
    print(f"  Columns: {df_raw.columns.tolist()}")
    
    # ENTSO-E exports may use either CET/CEST or UTC timestamp headers
    mtu_candidates = ['MTU (CET/CEST)', 'MTU (UTC)']
    price_candidates = ['Day-ahead Price (EUR/MWh)']
    
    mtu_col = next((col for col in mtu_candidates if col in df_raw.columns), None)
    price_col = next((col for col in price_candidates if col in df_raw.columns), None)
    
    if mtu_col is None or price_col is None:
        raise KeyError(
            f"Could not find expected ENTSO-E columns. Available columns: {df_raw.columns.tolist()}"
        )
    
    print(f"  Using MTU column: {mtu_col}")
    print(f"  Using price column: {price_col}")
    
    # Extract relevant columns
    df = df_raw[[mtu_col, price_col]].copy()
    df.columns = ['mtu', 'price']
    
    # Remove rows with missing values
    df = df.dropna()
    
    print(f"  After dropping NaN: {len(df)} rows")
    
    # Parse MTU column to extract start timestamp
    # Format: "01/01/2026 00:00:00 - 01/01/2026 00:15:00"
    def parse_mtu(mtu_str):
        try:
            # Extract start time (before the '-')
            start_str = mtu_str.split(' - ')[0].strip()
            # Parse as datetime
            return pd.to_datetime(start_str, format='%d/%m/%Y %H:%M:%S')
        except Exception as e:
            print(f"  Warning: Could not parse '{mtu_str}': {e}")
            return pd.NaT
    
    df['ts_utc'] = df['mtu'].apply(parse_mtu)
    
    # Remove rows where timestamp parsing failed
    df = df.dropna(subset=['ts_utc'])
    
    print(f"  After timestamp parsing: {len(df)} rows")
    
    # Convert to timezone-aware UTC
    df['ts_utc'] = df['ts_utc'].dt.tz_localize('UTC', ambiguous='infer', nonexistent='shift_forward')
    
    # Convert price to numeric and handle any non-numeric values
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    
    # Remove rows with invalid prices
    df = df.dropna(subset=['price'])
    
    # Convert EUR/MWh to EUR/kWh
    df['price'] = df['price'] / 1000.0
    
    # Select final columns
    df_clean = df[['ts_utc', 'price']].copy()
    
    # Sort by timestamp
    df_clean = df_clean.sort_values('ts_utc').reset_index(drop=True)
    
    # Remove duplicates (keep last)
    df_clean = df_clean.drop_duplicates(subset=['ts_utc'], keep='last')
    
    print(f"  Final clean data: {len(df_clean)} rows")
    print(f"  Date range: {df_clean['ts_utc'].min()} to {df_clean['ts_utc'].max()}")
    print(f"  Price range: {df_clean['price'].min():.6f} - {df_clean['price'].max():.6f} EUR/kWh")
    
    # Validate year
    actual_years = df_clean['ts_utc'].dt.year.unique()
    print(f"  Years in data: {sorted(actual_years)}")
    
    return df_clean

## 2. Process 2025 Data

In [3]:
# Process 2025 Excel file
excel_2025 = r'GUI_ENERGY_PRICES_202412312300-202512312300 (1).xlsx'
df_2025 = process_entsoe_excel(excel_2025, 2025)

# Add 2025-01-01 00:00:00 with the price from the first row
first_price_2025 = df_2025.iloc[0]['price']
first_row_2025 = pd.DataFrame({
    'ts_utc': [pd.Timestamp('2025-01-01 00:00:00', tz='UTC')],
    'price': [first_price_2025]
})
df_2025 = pd.concat([first_row_2025, df_2025], ignore_index=True)
print(f"Added 2025-01-01 00:00:00 entry with price {first_price_2025}")

# Save to parquet
output_2025 = 'entsoe_prices_2025.parquet'
df_2025.to_parquet(output_2025, index=False)
print(f"\n✓ Saved to {output_2025} ({len(df_2025)} records)")


Processing GUI_ENERGY_PRICES_202412312300-202512312300 (1).xlsx...
  Raw data shape: (35039, 2)
  Columns: ['01/01/2025 00:00:00 - 01/01/2025 00:15:00', '13.62']
  Time column: '01/01/2025 00:00:00 - 01/01/2025 00:15:00'
  Price column: '13.62'
  After dropping NaN: 35039 rows
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
    - passing `format` if your strings have a consistent format;
    - passin

## 3. Process 2026 Data

In [4]:
# Process 2026 CSV file
csv_2026 = r'Entsoe api\GUI_ENERGY_PRICES_202601010000-202701010000.csv'
df_2026 = process_entsoe_csv(csv_2026, 2026)

# Add 2026-01-01 00:00:00 with the price from the first row
first_price_2026 = df_2026.iloc[0]['price']
first_row_2026 = pd.DataFrame({
    'ts_utc': [pd.Timestamp('2026-01-01 00:00:00', tz='UTC')],
    'price': [first_price_2026]
})
df_2026 = pd.concat([first_row_2026, df_2026], ignore_index=True)
print(f"Added 2026-01-01 00:00:00 entry with price {first_price_2026}")

# Save to parquet
output_2026 = 'entsoe_prices_2026.parquet'
df_2026.to_parquet(output_2026, index=False)
print(f"\n✓ Saved to {output_2026} ({len(df_2026)} records)")


Processing Entsoe api\GUI_ENERGY_PRICES_202601010000-202701010000.csv...


FileNotFoundError: [Errno 2] No such file or directory: 'Entsoe api\\GUI_ENERGY_PRICES_202601010000-202701010000.csv'

## 4. Load and Merge Parquet Files

Combine both years into a single time series.

In [ ]:
# Load parquet files
df_2025_loaded = pd.read_parquet('entsoe_prices_2025.parquet')
df_2026_loaded = pd.read_parquet('entsoe_prices_2026.parquet')

print(f"Loaded 2025: {len(df_2025_loaded)} rows")
print(f"Loaded 2026: {len(df_2026_loaded)} rows")

# Merge both datasets
df_combined = pd.concat([df_2025_loaded, df_2026_loaded], ignore_index=True)

# Sort by timestamp
df_combined = df_combined.sort_values('ts_utc').reset_index(drop=True)

# Remove any duplicates (in case of overlap)
df_combined = df_combined.drop_duplicates(subset=['ts_utc'], keep='last')

print(f"\nCombined dataset: {len(df_combined)} rows")
print(f"Date range: {df_combined['ts_utc'].min()} to {df_combined['ts_utc'].max()}")
print(f"Price range: {df_combined['price'].min():.6f} - {df_combined['price'].max():.6f} EUR/kWh")

# Display summary statistics
print("\nSummary Statistics:")
print(df_combined['price'].describe())

## 5. Display Sample Data

In [ ]:
# Show first and last few rows
print("First 10 rows:")
display(df_combined.head(10))

print("\nLast 10 rows:")
display(df_combined.tail(10))

## 6. Plot Combined Time Series

Visualize the complete price data for 2025-2026.

In [ ]:
# Plot full time series
plt.figure(figsize=(16, 6))
plt.plot(df_combined['ts_utc'], df_combined['price'], linewidth=0.5, alpha=0.7)
plt.xlabel('Time (UTC)', fontsize=12)
plt.ylabel('Electricity Price (EUR/kWh)', fontsize=12)
plt.title('ENTSO-E Day-Ahead Electricity Prices - Netherlands (2025-2026)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Plotted {len(df_combined)} data points")

## 7. Monthly Statistics and Comparison

In [ ]:
# Add year and month columns for aggregation
df_combined['year'] = df_combined['ts_utc'].dt.year
df_combined['month'] = df_combined['ts_utc'].dt.month

# Calculate monthly statistics
monthly_stats = df_combined.groupby(['year', 'month'])['price'].agg([
    ('mean', 'mean'),
    ('median', 'median'),
    ('min', 'min'),
    ('max', 'max'),
    ('std', 'std')
]).reset_index()

print("Monthly Price Statistics (EUR/kWh):")
display(monthly_stats)

## 8. Monthly Average Price Plot

In [ ]:
# Create month labels
monthly_stats['month_label'] = pd.to_datetime(
    monthly_stats['year'].astype(str) + '-' + monthly_stats['month'].astype(str) + '-01'
)

# Plot monthly averages
fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(monthly_stats['month_label'], monthly_stats['mean'], width=20, alpha=0.7, label='Mean Price')
ax.plot(monthly_stats['month_label'], monthly_stats['median'], 'r-o', linewidth=2, markersize=6, label='Median Price')

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Price (EUR/kWh)', fontsize=12)
ax.set_title('Monthly Average Electricity Prices - Netherlands (2025-2026)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Daily Price Distribution (Histogram)

In [ ]:
# Plot histogram of prices
plt.figure(figsize=(12, 6))
plt.hist(df_combined['price'], bins=100, alpha=0.7, edgecolor='black')
plt.axvline(df_combined['price'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_combined["price"].mean():.5f} EUR/kWh')
plt.axvline(df_combined['price'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df_combined["price"].median():.5f} EUR/kWh')
plt.xlabel('Price (EUR/kWh)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Electricity Prices (2025-2026)', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 10. Save Individual and Combined Datasets to HomeOptimizer Input Folder

Save the 2025, 2026, and combined time series to the HomeOptimizer input folder for optimization.

In [ ]:
# Save datasets to current folder
datasets = {
    'entsoe_prices_2025.parquet': df_2025[['ts_utc', 'price']],
    'entsoe_prices_2026.parquet': df_2026[['ts_utc', 'price']],
    'entsoe_prices_2025_2026_combined.parquet': df_combined[['ts_utc', 'price']],
}
output_combined = 'entsoe_prices_2025_2026_combined.parquet'

for output_name, dataset in datasets.items():
    dataset.to_parquet(output_name, index=False)
    print(f"Saved {output_name} ({len(dataset)} records, {Path(output_name).stat().st_size / (1024*1024):.2f} MB)")
print(f"✓ Saved combined dataset to {output_combined}")
print(f"  Total records: {len(df_combined)}")
print(f"  File size: {Path(output_combined).stat().st_size / (1024*1024):.2f} MB")

# Also save to HomeOptimizer input folder
homeopt_dir = Path('../HomeOptimizer/input_data')
homeopt_dir.mkdir(parents=True, exist_ok=True)

homeopt_outputs = {
    'entsoe_prices_NL_2025.parquet': df_2025[['ts_utc', 'price']],
    'entsoe_prices_NL_2026.parquet': df_2026[['ts_utc', 'price']],
}

for output_name, dataset in homeopt_outputs.items():
    output_path = homeopt_dir / output_name
    dataset.to_parquet(output_path, index=False)
    print(f"\nSaved to HomeOptimizer input folder: {output_path}")
    print(f"  Records: {len(dataset)}")

homeopt_output = homeopt_dir / 'entsoe_prices_NL_2025_2026_combined.parquet'
df_combined[['ts_utc', 'price']].to_parquet(homeopt_output, index=False)
print(f"\n✓ Saved to HomeOptimizer input folder: {homeopt_output}")
print(f"  Ready for optimization!")
print(f"  Notebook will use: '../input_data/entsoe_prices_NL_2025_2026_combined.parquet'")